# DeepGuard: Deepfake Detection (Kaggle Version)
This notebook contains the complete DeepGuard pipeline, optimized for the Kaggle environment.


In [ ]:
!pip install facenet-pytorch pytorch-grad-cam torchvision streamlit pandas opencv-python matplotlib seaborn pyyaml pyngrok


## 1. Imports and Configuration


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm.notebook import tqdm
from collections import Counter
import logging

try:
    from facenet_pytorch import MTCNN
except ImportError:
    MTCNN = None
    
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    GradCAM = None

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Global Configuration parameters
config = {
    "SEED": SEED,
    "IMAGE_SIZE": 224,
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 2,
    "EPOCHS": 5, # Small number of epochs for sanity check
    "LEARNING_RATE": 0.001,
    "MODEL_PATH": "best_model.pth",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "NUM_CLASSES": 2,
    "CLASSES": ["REAL", "DEEPFAKE"]
}

print(f"Device: {config['DEVICE']}")

def get_logger(name="DeepGuard"):
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        ch = logging.StreamHandler()
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    return logger

logger = get_logger()



## 2. Dataset Discovery & Split
Automatically locate the dataset in `/kaggle/input/` and prepare splits.


In [ ]:
# Discover the dataset
all_images = []
all_labels = []

# Walk through Kaggle input directory
input_dir = "/kaggle/input"
if not os.path.exists(input_dir):
    print(f"Warning: {input_dir} not found. Running locally? Fallback to 'data/'")
    input_dir = "data/"
    os.makedirs(input_dir, exist_ok=True)

print("Discovering dataset...")
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            path = os.path.join(root, f)
            path_lower = path.lower()
            
            # REAL = 0, FAKE = 1
            if 'real' in path_lower:
                all_labels.append(0)
                all_images.append(path)
            elif 'fake' in path_lower:
                all_labels.append(1)
                all_images.append(path)

print(f"Total images discovered: {len(all_images)}")

# Check if dataset already has splits in directory structure
has_train = any('train' in p.lower() for p in all_images)
has_val = any('valid' in p.lower() or 'val' in p.lower() for p in all_images)
has_test = any('test' in p.lower() for p in all_images)

train_paths, train_labels = [], []
val_paths, val_labels = [], []
test_paths, test_labels = [], []

if has_train and has_val and has_test:
    print("Found existing train/val/test splits in folder structure!")
    for p, l in zip(all_images, all_labels):
        p_lower = p.lower()
        if 'train' in p_lower:
            train_paths.append(p)
            train_labels.append(l)
        elif 'valid' in p_lower or 'val' in p_lower:
            val_paths.append(p)
            val_labels.append(l)
        elif 'test' in p_lower:
            test_paths.append(p)
            test_labels.append(l)
else:
    print("No existing splits found, creating reproducible 80-10-10 stratified splits...")
    if len(all_images) > 0:
        # Split into 80% train and 20% temp
        train_paths, temp_paths, train_labels, temp_labels = train_test_split(
            all_images, all_labels, test_size=0.2, stratify=all_labels, random_state=config["SEED"]
        )
        # Split temp into 50% val and 50% test (10% and 10% of total)
        val_paths, test_paths, val_labels, test_labels = train_test_split(
            temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=config["SEED"]
        )

# Dataset Statistics
print("\n--- Dataset Statistics ---")
print(f"Total Images: {len(all_images)}")
print(f"Real Images: {all_labels.count(0)}")
print(f"Fake Images: {all_labels.count(1)}")
print(f"Training Count: {len(train_paths)}")
print(f"Validation Count: {len(val_paths)}")
print(f"Testing Count: {len(test_paths)}")



## 3. Dataset Inspection
Display a few samples to verify correctness.


In [ ]:
def show_samples(paths, labels, num_samples=4):
    if len(paths) == 0:
        return
        
    plt.figure(figsize=(15, 5))
    # Select a few random samples
    indices = np.random.choice(len(paths), min(num_samples, len(paths)), replace=False)
    
    for i, idx in enumerate(indices):
        path = paths[idx]
        label = labels[idx]
        
        try:
            img = Image.open(path).convert("RGB")
            plt.subplot(1, num_samples, i+1)
            plt.imshow(img)
            title = "REAL" if label == 0 else "DEEPFAKE"
            plt.title(f"{title}\n{os.path.basename(path)}")
            plt.axis('off')
        except Exception as e:
            print(f"Error loading {path}: {e}")
            
    plt.tight_layout()
    plt.show()

print("Sample Training Images:")
show_samples(train_paths, train_labels)



## 4. Preprocessing & PyTorch Dataset


In [ ]:
def get_train_transforms(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_val_transforms(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            logger.error(f"Error loading image {img_path}: {e}")
            # Fallback for corrupted images to avoid crashing training
            image = Image.new("RGB", (config["IMAGE_SIZE"], config["IMAGE_SIZE"]))
            
        if self.transform:
            image = self.transform(image)
            
        return image, label



## 5. Model Architecture & Face Detection


In [ ]:
class DeepfakeClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_backbone=True):
        super(DeepfakeClassifier, self).__init__()
        
        if pretrained:
            weights = EfficientNet_B0_Weights.DEFAULT
            self.backbone = efficientnet_b0(weights=weights)
        else:
            self.backbone = efficientnet_b0(weights=None)
            
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
                
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

    def unfreeze_backbone(self, num_layers=None):
        for param in self.backbone.parameters():
            param.requires_grad = True

class FaceDetector:
    def __init__(self, device=None):
        self.device = device if device else config["DEVICE"]
            
        if MTCNN is not None:
            self.detector = MTCNN(keep_all=False, device=self.device)
        else:
            logger.error("facenet_pytorch not installed. Face detection will fail.")
            self.detector = None

    def detect_and_crop(self, image):
        if self.detector is None:
            return None
            
        if isinstance(image, np.ndarray):
            image_pil = Image.fromarray(image)
        else:
            image_pil = image
            
        try:
            boxes, probs = self.detector.detect(image_pil)
            if boxes is None or len(boxes) == 0:
                return None
                
            box = boxes[0]
            width, height = image_pil.size
            x1, y1, x2, y2 = box
            
            w = x2 - x1
            h = y2 - y1
            
            x1 = max(0, x1 - 0.2 * w)
            y1 = max(0, y1 - 0.2 * h)
            x2 = min(width, x2 + 0.2 * w)
            y2 = min(height, y2 + 0.2 * h)
            
            return image_pil.crop((x1, y1, x2, y2))
        except Exception as e:
            logger.error(f"Face detection failed: {e}")
            return None



## 6. Sanity Check
Test data loading and forward pass before full training loop.


In [ ]:
# Sanity Check
print("--- Sanity Check ---")
if len(train_paths) > 0:
    test_ds = DeepfakeDataset(train_paths[:4], train_labels[:4], transform=get_train_transforms(config["IMAGE_SIZE"]))
    test_dl = DataLoader(test_ds, batch_size=2, shuffle=False)
    
    inputs, labels = next(iter(test_dl))
    print(f"Input batch shape: {inputs.shape}")
    print(f"Labels shape: {labels.shape}, values: {labels}")
    
    test_model = DeepfakeClassifier(num_classes=config["NUM_CLASSES"]).to(config["DEVICE"])
    inputs = inputs.to(config["DEVICE"])
    
    with torch.no_grad():
        outputs = test_model(inputs)
    
    print(f"Model output shape: {outputs.shape}")
    print("Sanity check passed! Model forward pass successful.")
else:
    print("No training data available for sanity check.")



## 7. Explainability & Inference Predictors


In [ ]:
class ImagePredictor:
    def __init__(self, model_path=None):
        self.device = config["DEVICE"]
        self.face_detector = FaceDetector(device=self.device)
        
        self.image_size = config["IMAGE_SIZE"]
        self.transforms = get_val_transforms(self.image_size)
        self.classes = config["CLASSES"]
        
        self.model = DeepfakeClassifier(num_classes=config["NUM_CLASSES"])
        
        model_path = model_path if model_path else config["MODEL_PATH"]
            
        if Path(model_path).exists():
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
            self.model = self.model.to(self.device)
            self.model.eval()
            logger.info("Predictor initialized successfully.")
        else:
            logger.warning(f"Model checkpoint not found at {model_path}. Predictions will be random untrained outputs.")
            self.model = self.model.to(self.device)
            self.model.eval()

    def predict_image(self, image_path_or_pil):
        if isinstance(image_path_or_pil, (str, Path)):
            try:
                image = Image.open(image_path_or_pil).convert("RGB")
            except Exception as e:
                logger.error(f"Error opening image: {e}")
                return {"error": str(e)}
        else:
            image = image_path_or_pil
            
        face = self.face_detector.detect_and_crop(image)
        if face is None:
            return {"error": "No face detected in the image."}
            
        input_tensor = self.transforms(face).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(input_tensor)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
            
        real_prob = float(probs[0])
        fake_prob = float(probs[1])
        
        if fake_prob > 0.5:
            prediction = "DEEPFAKE"
            confidence = fake_prob
        else:
            prediction = "REAL"
            confidence = real_prob
            
        return {
            "prediction": prediction,
            "confidence": confidence,
            "real_probability": real_prob,
            "fake_probability": fake_prob,
            "face_crop": face
        }

class GradCAMExplainer:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.transforms = get_val_transforms(config["IMAGE_SIZE"])
        
        if GradCAM is not None:
            try:
                target_layers = [model.backbone.features[-1]]
                self.cam = GradCAM(model=model, target_layers=target_layers)
            except AttributeError:
                logger.error("Could not find target layers for Grad-CAM.")
                self.cam = None
        else:
            self.cam = None
            
    def generate_heatmap(self, face_image_pil, target_class=1):
        if self.cam is None:
            return face_image_pil
            
        input_tensor = self.transforms(face_image_pil).unsqueeze(0)
        
        img_size = config["IMAGE_SIZE"]
        rgb_img = face_image_pil.resize((img_size, img_size))
        rgb_img = np.float32(rgb_img) / 255.0
        
        targets = [ClassifierOutputTarget(target_class)]
        
        grayscale_cam = self.cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        return Image.fromarray(visualization)



## 8. Training and Evaluation Functions


In [ ]:
def plot_history(history, save_path="training_history.png"):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

# Global variables to store the best metrics for the final summary
final_summary_metrics = {}

def train_model():
    device = config["DEVICE"]
    logger.info(f"Using device: {device}")
    
    if len(train_paths) == 0 or len(val_paths) == 0:
        logger.error("Dataset is empty.")
        return
        
    train_dataset = DeepfakeDataset(train_paths, train_labels, transform=get_train_transforms(config["IMAGE_SIZE"]))
    val_dataset = DeepfakeDataset(val_paths, val_labels, transform=get_val_transforms(config["IMAGE_SIZE"]))
    
    # Handle Class Imbalance using WeightedRandomSampler
    class_counts = Counter(train_labels)
    total_samples = len(train_labels)
    class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in train_labels]
    
    sampler = WeightedRandomSampler(sample_weights, num_samples=total_samples, replacement=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config["BATCH_SIZE"], sampler=sampler, num_workers=config["NUM_WORKERS"])
    val_loader = DataLoader(val_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, num_workers=config["NUM_WORKERS"])
    
    model = DeepfakeClassifier(num_classes=config["NUM_CLASSES"]).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    early_stopping_patience = 3
    epochs_no_improve = 0
    
    for epoch in range(config["EPOCHS"]):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['EPOCHS']} [Train]"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = 100. * correct / total
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config['EPOCHS']} [Val]"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = 100. * val_correct / val_total
        
        scheduler.step(epoch_val_loss)
        
        logger.info(f"Epoch [{epoch+1}/{config['EPOCHS']}] - Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), config["MODEL_PATH"])
            final_summary_metrics["Best_Val_Acc"] = epoch_val_acc
            logger.info(f"Saved best model to {config['MODEL_PATH']}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stopping_patience:
                logger.info(f"Early stopping triggered after {epoch+1} epochs.")
                break
                
    plot_history(history)
    return model

def evaluate_model():
    device = config["DEVICE"]
    
    if len(test_paths) == 0:
        logger.error("Test dataset is empty.")
        return
        
    test_dataset = DeepfakeDataset(test_paths, test_labels, transform=get_val_transforms(config["IMAGE_SIZE"]))
    test_loader = DataLoader(test_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, num_workers=config["NUM_WORKERS"])
    
    model = DeepfakeClassifier(num_classes=config["NUM_CLASSES"])
    if not Path(config["MODEL_PATH"]).exists():
        logger.error(f"Model checkpoint not found at {config['MODEL_PATH']}")
        return
        
    model.load_state_dict(torch.load(config["MODEL_PATH"], map_location=device))
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
            
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        roc_auc = 0.0
        
    logger.info(f"Accuracy:  {accuracy:.4f}")
    logger.info(f"Precision: {precision:.4f}")
    logger.info(f"Recall:    {recall:.4f}")
    logger.info(f"F1-score:  {f1:.4f}")
    logger.info(f"ROC-AUC:   {roc_auc:.4f}")
    
    final_summary_metrics["Test_Accuracy"] = accuracy * 100
    final_summary_metrics["Precision"] = precision
    final_summary_metrics["Recall"] = recall
    final_summary_metrics["F1"] = f1
    final_summary_metrics["ROC_AUC"] = roc_auc
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["REAL", "DEEPFAKE"], zero_division=0))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["REAL", "DEEPFAKE"], yticklabels=["REAL", "DEEPFAKE"])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()
    
    # ROC Curve Plot
    if roc_auc > 0:
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc="lower right")
        plt.show()



## 9. Run Training & Evaluation


In [ ]:
# Run training
train_model()



In [ ]:
# Run evaluation on test set
evaluate_model()



## 10. Final Summary


In [ ]:
print("\n" + "="*40)
print("          DeepGuard Dataset Summary          ")
print("="*40)
print(f"Total:      {len(all_images)}")
print(f"Real:       {all_labels.count(0)}")
print(f"Fake:       {all_labels.count(1)}")
print(f"Train:      {len(train_paths)}")
print(f"Validation: {len(val_paths)}")
print(f"Test:       {len(test_paths)}")
print("\nModel: EfficientNet-B0")

if "Best_Val_Acc" in final_summary_metrics:
    print(f"Best Validation Accuracy: {final_summary_metrics['Best_Val_Acc']:.2f}%")
if "Test_Accuracy" in final_summary_metrics:
    print(f"Test Accuracy:            {final_summary_metrics['Test_Accuracy']:.2f}%")
    print(f"Precision:                {final_summary_metrics['Precision']:.4f}")
    print(f"Recall:                   {final_summary_metrics['Recall']:.4f}")
    print(f"F1:                       {final_summary_metrics['F1']:.4f}")
    print(f"ROC-AUC:                  {final_summary_metrics['ROC_AUC']:.4f}")
print("="*40)



## 11. Streamlit App
(Optional) Run this block to generate and launch the Streamlit frontend. For Kaggle, localtunnel might be required.


In [ ]:
%%writefile app.py
import streamlit as st
import tempfile
import os
from PIL import Image
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms
import cv2
import numpy as np
import logging

try:
    from facenet_pytorch import MTCNN
except ImportError:
    MTCNN = None
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    GradCAM = None

# We must redefine the necessary classes here since this will run as a standalone script!
# ... (For brevity in this standalone app.py, we paste all required classes here) ...

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def get_val_transforms(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

class DeepfakeClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_backbone=True):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.backbone(x)

class FaceDetector:
    def __init__(self, device=None):
        self.device = device or get_device()
        self.detector = MTCNN(keep_all=False, device=self.device) if MTCNN else None
    def detect_and_crop(self, image):
        if not self.detector: return None
        image_pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
        try:
            boxes, _ = self.detector.detect(image_pil)
            if boxes is None or len(boxes) == 0: return None
            x1, y1, x2, y2 = boxes[0]
            w, h = x2 - x1, y2 - y1
            width, height = image_pil.size
            return image_pil.crop((max(0, x1 - 0.2 * w), max(0, y1 - 0.2 * h), min(width, x2 + 0.2 * w), min(height, y2 + 0.2 * h)))
        except: return None

class ImagePredictor:
    def __init__(self, model_path="best_model.pth"):
        self.device = get_device()
        self.face_detector = FaceDetector(device=self.device)
        self.transforms = get_val_transforms(224)
        self.model = DeepfakeClassifier(num_classes=2)
        if Path(model_path).exists():
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model = self.model.to(self.device).eval()
    def predict_image(self, image_path_or_pil):
        image = Image.open(image_path_or_pil).convert("RGB") if isinstance(image_path_or_pil, (str, Path)) else image_path_or_pil
        face = self.face_detector.detect_and_crop(image)
        if face is None: return {"error": "No face detected in the image."}
        input_tensor = self.transforms(face).unsqueeze(0).to(self.device)
        with torch.no_grad():
            outputs = self.model(input_tensor)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
        fake_prob = float(probs[1])
        return {"prediction": "DEEPFAKE" if fake_prob > 0.5 else "REAL", "confidence": fake_prob if fake_prob > 0.5 else float(probs[0]), "real_probability": float(probs[0]), "fake_probability": fake_prob, "face_crop": face}

class GradCAMExplainer:
    def __init__(self, model, device):
        self.transforms = get_val_transforms(224)
        self.cam = GradCAM(model=model, target_layers=[model.backbone.features[-1]]) if GradCAM else None
    def generate_heatmap(self, face_image_pil, target_class=1):
        if not self.cam: return face_image_pil
        input_tensor = self.transforms(face_image_pil).unsqueeze(0)
        rgb_img = np.float32(face_image_pil.resize((224, 224))) / 255.0
        cam_out = self.cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(target_class)])[0, :]
        return Image.fromarray(show_cam_on_image(rgb_img, cam_out, use_rgb=True))

st.set_page_config(page_title="DeepGuard - Deepfake Detection", page_icon="🛡️", layout="wide")
st.markdown("<style>.main-header{font-size:2.5rem;font-weight:700;color:#1E3A8A;margin-bottom:1rem;}</style>", unsafe_allow_html=True)

@st.cache_resource
def load_models():
    predictor = ImagePredictor()
    return predictor, None, GradCAMExplainer(predictor.model, predictor.device)

st.sidebar.title("🛡️ DeepGuard")
page = st.sidebar.radio("Navigation", ["Home", "Image Detection"])

try: image_predictor, _, explainer = load_models(); models_loaded = True
except Exception as e: st.sidebar.error(f"Error loading models: {e}"); models_loaded = False

if page == "Home":
    st.markdown('<p class="main-header">DeepGuard</p>', unsafe_allow_html=True)
    st.write("Welcome to DeepGuard. Uses transfer learning (EfficientNet-B0) to detect deepfakes.")
elif page == "Image Detection":
    st.markdown('<p class="main-header">Image Detection</p>', unsafe_allow_html=True)
    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])
    if uploaded_file and models_loaded:
        image = Image.open(uploaded_file).convert("RGB")
        col1, col2 = st.columns(2)
        with col1: st.image(image, use_container_width=True)
        with st.spinner("Analyzing..."): result = image_predictor.predict_image(image)
        with col2:
            if "error" in result: st.error(result["error"])
            else:
                st.success(f"PREDICTION: {result['prediction']}")
                st.metric("Confidence", f"{result['confidence'] * 100:.2f}%")
                if result.get("face_crop"):
                    heatmap = explainer.generate_heatmap(result["face_crop"], target_class=1 if result["prediction"] == "DEEPFAKE" else 0)
                    st.image(heatmap, caption="Grad-CAM", width=150)



In [ ]:
import urllib.request
# Run the Streamlit app
try:
    password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
    print("Password/Endpoint IP for localtunnel is:", password)
except:
    print("Could not retrieve IP. Run: !curl ipv4.icanhazip.com")
!streamlit run app.py & npx localtunnel --port 8501

